In [32]:
import numpy as np
import pandas as pd

In [33]:
train_df = pd.read_csv("./Models/data/train.csv")
test_df = pd.read_csv("./Models/data/test.csv")

target = 'Transport_Cost'

print(train_df.shape, test_df.shape)

(5000, 20) (500, 19)


In [34]:
train_df.isnull().sum().sort_values(ascending=False)

Transport_Method        1071
Equipment_Type           599
Supplier_Reliability     587
Rural_Hospital           586
Equipment_Weight         460
Equipment_Width          443
Equipment_Height         283
Supplier_Name              0
Hospital_Id                0
CrossBorder_Shipping       0
Base_Transport_Fee         0
Equipment_Value            0
Installation_Service       0
Urgent_Shipping            0
Fragile_Equipment          0
Hospital_Info              0
Order_Placed_Date          0
Delivery_Date              0
Hospital_Location          0
Transport_Cost             0
dtype: int64

In [35]:
# Missing values

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

missing_val_cols = train_df.columns[train_df.isnull().sum() > 0]
missing_df = train_df[missing_val_cols]

missing_categorical_cols = missing_df.select_dtypes(include=['object']).columns
missing_numerical_cols = missing_df.select_dtypes(exclude=['object']).columns

print("Categorical missing value columns:", missing_categorical_cols.tolist())
print("Filling missing values with 'Unknown' for categorical columns")

# Categorical missing values replaced with 'Unknown'
for col in missing_categorical_cols:
    train_df[col] = train_df[col].fillna('Unknown')
    test_df[col] = test_df[col].fillna('Unknown')

print("\nNumerical missing value columns:", missing_numerical_cols.tolist())

print("Filling missing Supplier_Reliability values with median")
median_supplier_reliability = train_df["Supplier_Reliability"].median()
train_df["Supplier_Reliability"] = train_df["Supplier_Reliability"].fillna(median_supplier_reliability)
test_df["Supplier_Reliability"] = test_df["Supplier_Reliability"].fillna(median_supplier_reliability)

print("Filling missing equipment values with iterative imputer")
equipment_cols = ['Equipment_Height', 'Equipment_Width', 'Equipment_Weight', 'Equipment_Value']
imputer = IterativeImputer(random_state=42)
train_df[equipment_cols] = imputer.fit_transform(train_df[equipment_cols])
test_df[equipment_cols] = imputer.transform(test_df[equipment_cols])

Categorical missing value columns: ['Equipment_Type', 'Transport_Method', 'Rural_Hospital']
Filling missing values with 'Unknown' for categorical columns

Numerical missing value columns: ['Supplier_Reliability', 'Equipment_Height', 'Equipment_Width', 'Equipment_Weight']
Filling missing Supplier_Reliability values with median
Filling missing equipment values with iterative imputer


In [36]:
# Right skewed target feature

y_train = train_df[target]
train_df.drop([target], axis=1, inplace=True)

min_cost = y_train.min()
shift_value = abs(min_cost) + 1
y_train_log = np.log1p(y_train + shift_value)


In [37]:
# Combining both the train and test data to do feature engineering, preprocessing

joined_df = pd.concat([train_df, test_df], axis=0, ignore_index=True)
joined_df.shape

(5500, 19)

In [38]:
import re

def parse_location(location_str):
    """Parse hospital location to extract state, zip, and military status"""
    if pd.isna(location_str) or location_str == 'Missing':
        return 'Missing', 'Missing', 0
    
    # Check for military addresses (APO, FPO, DPO with AA, AE, AP)
    is_military = 1 if re.search(r'\b(APO|FPO|DPO)\b', str(location_str), re.IGNORECASE) else 0
    is_military = is_military or (1 if re.search(r'\b(AA|AE|AP)\b', str(location_str)) else 0)
    is_military = "Yes" if is_military else "No"
    
    # Extract state (2 letter code)
    state_match = re.search(r'\b([A-Z]{2})\b\s+\d{5}', str(location_str))
    if state_match:
        state = state_match.group(1)
    else:
        state = 'Missing'
    
    # Extract zip code (5 digits)
    zip_match = re.search(r'\b(\d{5})\b', str(location_str))
    if zip_match:
        zip_code = zip_match.group(1)
    else:
        zip_code = 'Missing'
    
    return state, zip_code, is_military

location_data = joined_df['Hospital_Location'].apply(parse_location)
joined_df['Location_State'] = location_data.apply(lambda x: x[0])
joined_df['Location_Zip'] = location_data.apply(lambda x: x[1])
joined_df['Location_Is_Military'] = location_data.apply(lambda x: x[2])

In [39]:
# Date features

joined_df['Order_Placed_Date'] = pd.to_datetime(joined_df['Order_Placed_Date'], format='%m/%d/%y')
joined_df['Delivery_Date'] = pd.to_datetime(joined_df['Delivery_Date'], format='%m/%d/%y')

mask = joined_df['Delivery_Date'] < joined_df['Order_Placed_Date']
joined_df.loc[mask, ['Order_Placed_Date', 'Delivery_Date']] = joined_df.loc[mask, ['Delivery_Date', 'Order_Placed_Date']].values

joined_df['Order_Year'] = joined_df['Order_Placed_Date'].dt.year
joined_df['Order_Month'] = joined_df['Order_Placed_Date'].dt.month
joined_df['Order_Day'] = joined_df['Order_Placed_Date'].dt.day
joined_df['Order_Dow'] = joined_df['Order_Placed_Date'].dt.dayofweek
joined_df['Order_Quarter'] = joined_df['Order_Placed_Date'].dt.quarter

joined_df['Delivery_Year'] = joined_df['Delivery_Date'].dt.year
joined_df['Delivery_Month'] = joined_df['Delivery_Date'].dt.month
joined_df['Delivery_Day'] = joined_df['Delivery_Date'].dt.day
joined_df['Delivery_Dow'] = joined_df['Delivery_Date'].dt.dayofweek
joined_df['Delivery_Quarter'] = joined_df['Delivery_Date'].dt.quarter

joined_df['Delivery_Time'] = (joined_df['Delivery_Date'] - joined_df['Order_Placed_Date']).dt.days

In [40]:
# Equipment features

joined_df['Equipment_Area'] = joined_df['Equipment_Height'] * joined_df['Equipment_Width']
joined_df['Equipment_Density'] = joined_df['Equipment_Weight'] / joined_df['Equipment_Area']
joined_df['Equipment_Cost_per_Area'] = joined_df['Equipment_Value'] / joined_df['Equipment_Area']
joined_df['Equipment_Cost_per_Weight'] = joined_df['Equipment_Value'] / joined_df['Equipment_Weight']

In [41]:
# Dropping unneccesary cols
drop_cols = ['Hospital_Id', 'Hospital_Location', 'Supplier_Name', 'Order_Placed_Date', 'Delivery_Date']
joined_df.drop(drop_cols, axis=1, inplace=True)

In [42]:
joined_df.select_dtypes(include=['object']).nunique().sort_values(ascending=False)

Location_Zip            5368
Location_State            54
Equipment_Type             8
Transport_Method           4
Rural_Hospital             3
Urgent_Shipping            2
CrossBorder_Shipping       2
Hospital_Info              2
Fragile_Equipment          2
Installation_Service       2
Location_Is_Military       2
dtype: int64

In [43]:
# Scaling numerical features


In [44]:
# Encoding categorical features

# location_cols = ['Location_State', 'Location_Zip']
location_cols = joined_df.select_dtypes(include=['object']).columns.tolist()
print("Label encoding Location features: ", location_cols)

for col in location_cols:
    unique_vals = joined_df[col].unique().tolist()
    val_to_int = {val: idx for idx, val in enumerate(sorted(unique_vals))}
    joined_df[col] = joined_df[col].map(val_to_int)

# categorical_cols = joined_df.select_dtypes(include=['object']).columns.tolist()
# categorical_cols = [col for col in categorical_cols if col not in location_cols]
# print("One-hot encoding categorical features: ", categorical_cols)

# joined_df = pd.get_dummies(joined_df, columns=categorical_cols, drop_first=True)

Label encoding Location features:  ['Equipment_Type', 'CrossBorder_Shipping', 'Urgent_Shipping', 'Installation_Service', 'Transport_Method', 'Fragile_Equipment', 'Hospital_Info', 'Rural_Hospital', 'Location_State', 'Location_Zip', 'Location_Is_Military']


In [45]:
train_processed = joined_df.iloc[:train_df.shape[0], :].copy()
test_processed = joined_df.iloc[train_df.shape[0]:, :].copy()

test_processed.insert(0, "Hospital_Id", test_df["Hospital_Id"].values)

train_processed["Transport_Cost"] = y_train
train_processed["Transport_Cost_Log"] = y_train_log
train_processed["Target_Shift_Value"] = shift_value

print("Final train shape:", train_processed.shape)
print("Final test shape:", test_processed.shape)

Final train shape: (5000, 35)
Final test shape: (500, 33)


In [46]:
train_processed.to_csv("./Models/data/train_processed.csv", index=False)
test_processed.to_csv("./Models/data/test_processed.csv", index=False)